# 19. Registro de autoridades / funciones adicionales

**Fuente:** `data/raw/registro_autoridades.csv`
**Salida:** `data/processed/registro_autoridades.csv`

Cargos/designaciones adicionales ejercidas en paralelo al contrato laboral
(Director, Coordinador, membresias de consejo, subrogaciones). Este notebook
solo hace limpieza basica (tipado, strings, calidad de datos) - la
integracion sustantiva contra `historial_laboral_categoria_cargo`
(`procesar_registro_autoridades`/`construir_features_funciones_adicionales`,
ver DEC-012 en `context/DECISION_LOG.md`) se hace en `04_trayectorias.ipynb`,
porque depende de `CATEGORIA_CARGO` (calculada ahi).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('registro_autoridades.csv', low_memory=False)
df.head()

## 2. Exploracion inicial

In [ ]:
pc.resumen(df, 'registro_autoridades')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df)

## 4. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['FECHADESDE', 'FECHAHASTA'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDESTRUCTURAORGANICA'])

## 5. Diagnostico de calidad de datos

`FECHAHASTA` anterior a `FECHADESDE` (rango invertido) y filas duplicadas
exactas - no se corrigen ni se eliminan aqui (se conservan para
trazabilidad), solo se documentan; `procesar_registro_autoridades` en
`04_trayectorias.ipynb` las marca con `ES_RUIDO_CALIDAD_DATOS`/
`ES_DUPLICADO_EXACTO` y las excluye de los conteos por persona.

In [ ]:
malas_fechas = df['FECHAHASTA'].notna() & (df['FECHAHASTA'] < df['FECHADESDE'])
print(f"Filas con FECHAHASTA < FECHADESDE: {malas_fechas.sum()} ({malas_fechas.mean()*100:.1f}%)")

dup = df.duplicated(subset=['IDPERSONA', 'IDESTRUCTURAORGANICA', 'TIPOAUTORIDAD', 'FECHADESDE', 'FECHAHASTA'], keep=False)
print(f"Filas duplicadas exactas: {dup.sum()} ({dup.mean()*100:.1f}%)")

## Graficos exploratorios

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Designaciones por año de inicio**.

In [ ]:
pc.grafico_por_anio(df['FECHADESDE'], 'Designaciones de autoridad/funcion por año')

**Tipos de autoridad/funcion mas frecuentes** (incluye "Subrogación" como categoria propia).

In [ ]:
pc.grafico_barras(df['TIPOAUTORIDAD'], 'Tipos de autoridad/funcion', top=15)

## 7. Verificacion final

In [ ]:
pc.resumen(df, 'registro_autoridades (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'registro_autoridades.csv')